In [1]:
import sys
sys.path.insert(1, '../')

In [2]:
import joblib
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import precision_recall_curve, roc_curve

from functions.data import load_dataset
from functions.data import DATASETS_LS as datasets
from functions.analysis import create_df, create_fold_df
from functions.statistical_tests import apply_friedman_test, apply_nemenyi_test
from functions.plotting import compute_ylim, add_significance_bars


In [3]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="The total space of parameters")
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

## Load results

In [4]:
filename = '../models/ensembles/results'
with open(filename, 'rb') as f:
    classic_dict = pickle.load(f)

filename = '../models/ensembles/results_folds'
with open(filename, 'rb') as f:
    classic_folds_dict = pickle.load(f)

filename = '../models/special-ensembles/results'
with open(filename, 'rb') as f:
    ens_dict = pickle.load(f)

filename = '../models/special-ensembles/results_folds'
with open(filename, 'rb') as f:
    ens_folds_dict = pickle.load(f)

scores_dict = {
    key: {**classic_dict[key], **ens_dict[key]}  # Merge inner dicts
    for key in classic_dict
}

folds_dict = {
    key: {**classic_folds_dict[key], **ens_folds_dict[key]}  # Merge inner dicts
    for key in classic_folds_dict
}


In [5]:
models = [
      "rf", "ada", "gbm", "cat", "lgbm", "xgb", "rusboost", "easyEnsemble", "balancedRF",
      ]

In [6]:
# test function
df = create_df(scores_dict, "oil", models)
df

,roc,roc_std,ap,ap_std,precision,precision_std,recall,recall_std,f1_score,f1_std,mcc,mcc_std,ba,ba_std,brier,brier_std,gmean,gmean_std,thresh,tresh_std
rf,0.954913,0.022165,0.611664,0.097060,0.722500,0.161924,0.703788,0.122399,0.687619,0.041360,0.682910,0.044886,0.926913,0.033283,0.037129,0.011183,0.926353,0.033319,0.206481,0.038896
ada,0.975267,0.006724,0.712369,0.103320,0.811111,0.118374,0.712121,0.038030,0.755938,0.067371,0.749647,0.071642,0.957554,0.003664,0.167357,0.002936,0.956605,0.003832,0.493193,0.011129
gbm,0.970079,0.006198,0.709911,0.093447,0.854978,0.125407,0.646717,0.063174,0.732781,0.074312,0.728965,0.082739,0.944409,0.012350,0.030416,0.007570,0.942680,0.013109,0.647669,0.387881
cat,0.965166,0.003736,0.729943,0.057918,0.852137,0.169602,0.696717,0.108716,0.746032,0.072392,0.748130,0.074906,0.933012,0.011904,0.027421,0.008653,0.931585,0.011854,0.541895,0.301408
lgbm,0.938354,0.018582,0.689647,0.083522,0.845238,0.140052,0.688384,0.079673,0.748779,0.067107,0.744900,0.074590,0.917043,0.023256,0.029256,0.011446,0.914585,0.026115,0.438034,0.367620
xgb,0.964190,0.010220,0.613956,0.101991,0.656439,0.060317,0.719192,0.112405,0.680555,0.070376,0.666395,0.067906,0.936410,0.013476,0.032438,0.010095,0.934136,0.014503,0.277016,0.164331
rusboost,0.935299,0.017958,0.542801,0.129300,0.763095,0.138341,0.688384,0.079673,0.710702,0.069851,0.704633,0.065326,0.885343,0.031919,0.135230,0.004922,0.879482,0.034339,0.848170,0.011911
easyEnsemble,0.931111,0.017877,0.420518,0.117258,0.477016,0.128173,0.748232,0.084162,0.565611,0.088206,0.561456,0.068041,0.897897,0.029318,0.170128,0.002550,0.896056,0.030983,0.665528,0.045979
balancedRF,0.933683,0.015242,0.458096,0.111086,0.650275,0.143352,0.661869,0.112589,0.643596,0.090549,0.631746,0.093707,0.884454,0.023154,0.125944,0.004934,0.880169,0.024028,0.840282,0.026635


## Datasets

In [7]:
datasets

['abalone_19',
 'car_eval_4',
 'coil_2000',
 'ecoli',
 'isolet',
 'letter_img',
 'libras_move',
 'mammography',
 'oil',
 'optical_digits',
 'ozone_level',
 'pen_digits',
 'protein_homo',
 'satimage',
 'scene',
 'sick_euthyroid',
 'solar_flare_m0',
 'spectrometer',
 'thyroid_sick',
 'us_crime',
 'webpage',
 'wine_quality',
 'yeast_me2',
 'cleveland-0_vs_4',
 'glass-0-1-4-6_vs_2',
 'led7digit-0-2-4-5-6-7-8-9_vs_1',
 'page-blocks-1-3_vs_4',
 'pima',
 'poker-8-9_vs_5',
 'diabetes130',
 'default_credit',
 'htru2',
 'credit_fraud',
 'secom',
 'bank-marketing',
 'telco',
 'adult']

In [8]:
len(datasets)

37

In [ ]:
# rf and gbm (from notebook 01-analysis-of-ensembles) are included alongside
# the resampling ensembles for comparison
prob_dist_models = ["rf", "gbm", "rusboost", "easyEnsemble", "balancedRF"]

PROB_DIST_MODEL_DIRS = {
    "rf": "../models/ensembles",
    "gbm": "../models/ensembles",
    "rusboost": "../models/special-ensembles",
    "easyEnsemble": "../models/special-ensembles",
    "balancedRF": "../models/special-ensembles",
}

n_datasets = len(datasets)
n_estimators = len(prob_dist_models)

fig, axes = plt.subplots(n_datasets, n_estimators,
                         figsize=(4 * n_estimators, 3 * n_datasets))

for i, dataset in enumerate(datasets):
    _, X_test, _, y_test = load_dataset(dataset)

    for j, estimator in enumerate(prob_dist_models):
        search = joblib.load(f"{PROB_DIST_MODEL_DIRS[estimator]}/{dataset}_{estimator}.pkl")
        probs = search.predict_proba(X_test)[:, 1]

        ax = axes[i, j]
        ax.hist(probs, bins=30, density=True, color="steelblue")
        ax.set_xlim(0, 1)
        stats_text = (
            f"min={probs.min():.2f}\n"
            f"max={probs.max():.2f}\n"
            f"mean={probs.mean():.2f}\n"
            f"p75={np.percentile(probs, 75):.2f}\n"
            f"p25={np.percentile(probs, 25):.2f}"
        )
        ax.text(0.97, 0.97, stats_text,
                transform=ax.transAxes,
                fontsize=7,
                verticalalignment="top",
                horizontalalignment="right",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="gray", alpha=0.8))
        ax.set_title(f"{dataset}\n{estimator}", fontsize=9)

        if j == 0:
            ax.set_ylabel("Density", fontsize=9)
        if i == n_datasets - 1:
            ax.set_xlabel("P(class = 1)", fontsize=9)

plt.suptitle("Predicted Probability of Class 1", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../figures/special-ensembles_prob_dist.png', dpi=900, bbox_inches='tight')
plt.show()

## ROC-AUC Friedman test

In [ ]:
friedman_rows = []
nemenyi_dict = {}

for data in datasets:
    df = create_fold_df(folds_dict, data, "roc")
    statistic, pvalue = apply_friedman_test(df)
    friedman_rows.append({"dataset": data, "statistic": statistic, "pvalue": pvalue})

    # follow up with Nemenyi post-hoc test only when Friedman is significant
    if pvalue < 0.05:
        nemenyi_dict[data] = apply_nemenyi_test(df)

friedman_df = pd.DataFrame(friedman_rows).set_index("dataset")
friedman_df

## Plot ROC-AUC

In [ ]:
# Create figure with 37 subplots
fig, axes = plt.subplots(nrows=10, ncols=4, figsize=(24, 50))  # 10x4 grid for 37 plots (with 3 empty)
axes = axes.ravel()  # Flatten for easy iteration

# Loop through each DataFrame
for i, data in enumerate(datasets):
    df = create_df(scores_dict, data, models)
    
    ax = axes[i]
    
    # Extract data
    cls = df.index
    means = df.iloc[:, 0]
    stds = df.iloc[:, 1]

    upper = df.iloc[0, 0] + df.iloc[0, 1]
    lower = df.iloc[0, 0] - df.iloc[0, 1]

    # Determine significant comparisons (if any) up front, so we can
    # reserve extra headroom above the data for the significance bars
    best_model = None
    sig_models = []
    if data in nemenyi_dict:
        best_model = means.idxmax()
        posthoc = nemenyi_dict[data]
        sig_models = [
            m for m in models
            if m != best_model and posthoc.loc[best_model, m] < 0.05
        ]

    # Create errorbar plot
    ax.errorbar(x=cls, y=means, yerr=stds,
                fmt='o', capsize=5, markersize=8)

    # Customize subplot
    ax.set_title(data, fontsize=20)
    ylims = compute_ylim(means, stds, upper, lower)
    if sig_models:
        # reserve headroom above the data for the significance bars
        y0, y1 = ylims
        ylims = (y0, y1 + (y1 - y0) * 0.4)
    ax.set_ylim(ylims)
    ax.tick_params(axis='x', rotation=90, labelsize=14)
    ax.grid(True, alpha=0.3)

    # Add horizontal lines at RF error bars
    ax.axhline(y=upper, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=lower, color='r', linestyle='--', alpha=0.5)

    # Color area between RF error bars
    ax.axhspan(ymin=lower, ymax=upper, facecolor='pink', alpha=0.3)

    # If the Friedman test was significant, draw Nemenyi significance bars
    # from the best performing model to each model significantly worse than it
    if sig_models:
        data_max = (means + stds).max()
        add_significance_bars(ax, models, data_max, best_model, sig_models)

# Hide unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.suptitle(f'Model Performance Across {len(datasets)} Datasets (ROC-AUC mean ± std)', fontsize=20, y=1.02)
plt.savefig('../figures/special-ensembles_rocauc.png', dpi=300, bbox_inches='tight')
plt.show()

## Average Precision Friedman test

In [ ]:
friedman_rows = []
nemenyi_dict_ap = {}

for data in datasets:
    df = create_fold_df(folds_dict, data, "ap")
    statistic, pvalue = apply_friedman_test(df)
    friedman_rows.append({"dataset": data, "statistic": statistic, "pvalue": pvalue})

    # follow up with Nemenyi post-hoc test only when Friedman is significant
    if pvalue < 0.05:
        nemenyi_dict_ap[data] = apply_nemenyi_test(df)

friedman_df_ap = pd.DataFrame(friedman_rows).set_index("dataset")
friedman_df_ap

## Plot average precision

In [ ]:
# Create figure with 37 subplots
fig, axes = plt.subplots(nrows=10, ncols=4, figsize=(24, 50))  # 10x4 grid for 37 plots (with 3 empty)
axes = axes.ravel()  # Flatten for easy iteration

# Loop through each DataFrame
for i, data in enumerate(datasets):
    df = create_df(scores_dict, data, models)
    
    ax = axes[i]
    
    # Extract data
    cls = df.index
    means = df.iloc[:, 2]
    stds = df.iloc[:, 3]

    upper = df.iloc[0, 2] + df.iloc[0, 3]
    lower = df.iloc[0, 2] - df.iloc[0, 3]

    # Determine significant comparisons (if any) up front, so we can
    # reserve extra headroom above the data for the significance bars
    best_model = None
    sig_models = []
    if data in nemenyi_dict_ap:
        best_model = means.idxmax()
        posthoc = nemenyi_dict_ap[data]
        sig_models = [
            m for m in models
            if m != best_model and posthoc.loc[best_model, m] < 0.05
        ]

    # Create errorbar plot
    ax.errorbar(x=cls, y=means, yerr=stds,
                fmt='o', capsize=5, markersize=8)

    # Customize subplot
    ax.set_title(data, fontsize=20)
    ylims = compute_ylim(means, stds, upper, lower)
    if sig_models:
        # reserve headroom above the data for the significance bars
        y0, y1 = ylims
        ylims = (y0, y1 + (y1 - y0) * 0.4)
    ax.set_ylim(ylims)
    ax.tick_params(axis='x', rotation=90, labelsize=14)
    ax.grid(True, alpha=0.3)

    # Add horizontal lines at RF error bars
    ax.axhline(y=upper, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=lower, color='r', linestyle='--', alpha=0.5)

    # Color area between RF error bars
    ax.axhspan(ymin=lower, ymax=upper, facecolor='pink', alpha=0.3)

    # If the Friedman test was significant, draw Nemenyi significance bars
    # from the best performing model to each model significantly worse than it
    if sig_models:
        data_max = (means + stds).max()
        add_significance_bars(ax, models, data_max, best_model, sig_models)

# Hide unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.suptitle(f'Model Performance Across {len(datasets)} Datasets (Average Precision)', fontsize=20, y=1.02)
plt.savefig('../figures/special-ensembles_ap.png', dpi=300, bbox_inches='tight')
plt.show()

## Recall Friedman test

In [ ]:
friedman_rows = []
nemenyi_dict_recall = {}

for data in datasets:
    df = create_fold_df(folds_dict, data, "recall")
    statistic, pvalue = apply_friedman_test(df)
    friedman_rows.append({"dataset": data, "statistic": statistic, "pvalue": pvalue})

    # follow up with Nemenyi post-hoc test only when Friedman is significant
    if pvalue < 0.05:
        nemenyi_dict_recall[data] = apply_nemenyi_test(df)

friedman_df_recall = pd.DataFrame(friedman_rows).set_index("dataset")
friedman_df_recall

## Plot Recall

In [ ]:
# Create figure with 37 subplots
fig, axes = plt.subplots(nrows=10, ncols=4, figsize=(24, 50))  # 10x4 grid for 37 plots (with 3 empty)
axes = axes.ravel()  # Flatten for easy iteration

# Loop through each DataFrame
for i, data in enumerate(datasets):
    df = create_df(scores_dict, data, models)
    
    ax = axes[i]
    
    # Extract data
    cls = df.index
    means = df.iloc[:, 6]
    stds = df.iloc[:, 7]

    upper = df.iloc[0, 6] + df.iloc[0, 7]
    lower = df.iloc[0, 6] - df.iloc[0, 7]

    # Determine significant comparisons (if any) up front, so we can
    # reserve extra headroom above the data for the significance bars
    best_model = None
    sig_models = []
    if data in nemenyi_dict_recall:
        best_model = means.idxmax()
        posthoc = nemenyi_dict_recall[data]
        sig_models = [
            m for m in models
            if m != best_model and posthoc.loc[best_model, m] < 0.05
        ]

    # Create errorbar plot
    ax.errorbar(x=cls, y=means, yerr=stds,
                fmt='o', capsize=5, markersize=8)

    # Customize subplot
    ax.set_title(data, fontsize=20)
    ylims = compute_ylim(means, stds, upper, lower)
    if sig_models:
        # reserve headroom above the data for the significance bars
        y0, y1 = ylims
        ylims = (y0, y1 + (y1 - y0) * 0.4)
    ax.set_ylim(ylims)
    ax.tick_params(axis='x', rotation=90, labelsize=14)
    ax.grid(True, alpha=0.3)

    # Add horizontal lines at RF error bars
    ax.axhline(y=upper, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=lower, color='r', linestyle='--', alpha=0.5)

    # Color area between RF error bars
    ax.axhspan(ymin=lower, ymax=upper, facecolor='pink', alpha=0.3)

    # If the Friedman test was significant, draw Nemenyi significance bars
    # from the best performing model to each model significantly worse than it
    if sig_models:
        data_max = (means + stds).max()
        add_significance_bars(ax, models, data_max, best_model, sig_models)

# Hide unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.suptitle(f'Model Performance Across {len(datasets)} Datasets (Average Recall)', fontsize=20, y=1.02)
plt.savefig('../figures/special-ensembles_recall.png', dpi=300, bbox_inches='tight')
plt.show()

## F1-score Friedman test

In [ ]:
friedman_rows = []
nemenyi_dict_f1 = {}

for data in datasets:
    df = create_fold_df(folds_dict, data, "f1")
    statistic, pvalue = apply_friedman_test(df)
    friedman_rows.append({"dataset": data, "statistic": statistic, "pvalue": pvalue})

    # follow up with Nemenyi post-hoc test only when Friedman is significant
    if pvalue < 0.05:
        nemenyi_dict_f1[data] = apply_nemenyi_test(df)

friedman_df_f1 = pd.DataFrame(friedman_rows).set_index("dataset")
friedman_df_f1

## Plot F1-score

In [ ]:
# Create figure with 37 subplots
fig, axes = plt.subplots(nrows=10, ncols=4, figsize=(24, 50))  # 10x4 grid for 37 plots (with 3 empty)
axes = axes.ravel()  # Flatten for easy iteration

# Loop through each DataFrame
for i, data in enumerate(datasets):
    df = create_df(scores_dict, data, models)
    
    ax = axes[i]
    
    # Extract data
    cls = df.index
    means = df.iloc[:, 8]
    stds = df.iloc[:, 9]

    upper = df.iloc[0, 8] + df.iloc[0, 9]
    lower = df.iloc[0, 8] - df.iloc[0, 9]

    # Determine significant comparisons (if any) up front, so we can
    # reserve extra headroom above the data for the significance bars
    best_model = None
    sig_models = []
    if data in nemenyi_dict_f1:
        best_model = means.idxmax()
        posthoc = nemenyi_dict_f1[data]
        sig_models = [
            m for m in models
            if m != best_model and posthoc.loc[best_model, m] < 0.05
        ]

    # Create errorbar plot
    ax.errorbar(x=cls, y=means, yerr=stds,
                fmt='o', capsize=5, markersize=8)

    # Customize subplot
    ax.set_title(data, fontsize=20)
    ylims = compute_ylim(means, stds, upper, lower)
    if sig_models:
        # reserve headroom above the data for the significance bars
        y0, y1 = ylims
        ylims = (y0, y1 + (y1 - y0) * 0.4)
    ax.set_ylim(ylims)
    ax.tick_params(axis='x', rotation=90, labelsize=14)
    ax.grid(True, alpha=0.3)

    # Add horizontal lines at RF error bars
    ax.axhline(y=upper, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=lower, color='r', linestyle='--', alpha=0.5)

    # Color area between RF error bars
    ax.axhspan(ymin=lower, ymax=upper, facecolor='pink', alpha=0.3)

    # If the Friedman test was significant, draw Nemenyi significance bars
    # from the best performing model to each model significantly worse than it
    if sig_models:
        data_max = (means + stds).max()
        add_significance_bars(ax, models, data_max, best_model, sig_models)

# Hide unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.suptitle(f'Model Performance Across {len(datasets)} Datasets (F1-score)', fontsize=20, y=1.02)
plt.savefig('../figures/special-ensembles_f1-score.png', dpi=300, bbox_inches='tight')
plt.show()

## MCC Friedman test

In [ ]:
friedman_rows = []
nemenyi_dict_mcc = {}

for data in datasets:
    df = create_fold_df(folds_dict, data, "mcc")
    statistic, pvalue = apply_friedman_test(df)
    friedman_rows.append({"dataset": data, "statistic": statistic, "pvalue": pvalue})

    # follow up with Nemenyi post-hoc test only when Friedman is significant
    if pvalue < 0.05:
        nemenyi_dict_mcc[data] = apply_nemenyi_test(df)

friedman_df_mcc = pd.DataFrame(friedman_rows).set_index("dataset")
friedman_df_mcc

## MCC

In [ ]:
# Create figure with 37 subplots
fig, axes = plt.subplots(nrows=10, ncols=4, figsize=(24, 50))  # 10x4 grid for 37 plots (with 3 empty)
axes = axes.ravel()  # Flatten for easy iteration

# Loop through each DataFrame
for i, data in enumerate(datasets):
    df = create_df(scores_dict, data, models)
    
    ax = axes[i]
    
    # Extract data
    cls = df.index
    means = df.iloc[:, 10]
    stds = df.iloc[:, 11]

    upper = df.iloc[0, 10] + df.iloc[0, 11]
    lower = df.iloc[0, 10] - df.iloc[0, 11]

    # Determine significant comparisons (if any) up front, so we can
    # reserve extra headroom above the data for the significance bars
    best_model = None
    sig_models = []
    if data in nemenyi_dict_mcc:
        best_model = means.idxmax()
        posthoc = nemenyi_dict_mcc[data]
        sig_models = [
            m for m in models
            if m != best_model and posthoc.loc[best_model, m] < 0.05
        ]

    # Create errorbar plot
    ax.errorbar(x=cls, y=means, yerr=stds,
                fmt='o', capsize=5, markersize=8)

    # Customize subplot
    ax.set_title(data, fontsize=20)
    ylims = compute_ylim(means, stds, upper, lower)
    if sig_models:
        # reserve headroom above the data for the significance bars
        y0, y1 = ylims
        ylims = (y0, y1 + (y1 - y0) * 0.4)
    ax.set_ylim(ylims)
    ax.tick_params(axis='x', rotation=90, labelsize=14)
    ax.grid(True, alpha=0.3)

    # Add horizontal lines at RF error bars
    ax.axhline(y=upper, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=lower, color='r', linestyle='--', alpha=0.5)

    # Color area between RF error bars
    ax.axhspan(ymin=lower, ymax=upper, facecolor='pink', alpha=0.3)

    # If the Friedman test was significant, draw Nemenyi significance bars
    # from the best performing model to each model significantly worse than it
    if sig_models:
        data_max = (means + stds).max()
        add_significance_bars(ax, models, data_max, best_model, sig_models)

# Hide unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.suptitle(f'Model Performance Across {len(datasets)} Datasets (MCC)', fontsize=20, y=1.02)
plt.savefig('../figures/special-ensembles_mcc.png', dpi=300, bbox_inches='tight')
plt.show()

## Balanced Accuracy Friedman test

In [ ]:
friedman_rows = []
nemenyi_dict_ba = {}

for data in datasets:
    df = create_fold_df(folds_dict, data, "ba")
    statistic, pvalue = apply_friedman_test(df)
    friedman_rows.append({"dataset": data, "statistic": statistic, "pvalue": pvalue})

    # follow up with Nemenyi post-hoc test only when Friedman is significant
    if pvalue < 0.05:
        nemenyi_dict_ba[data] = apply_nemenyi_test(df)

friedman_df_ba = pd.DataFrame(friedman_rows).set_index("dataset")
friedman_df_ba

## Balanced accuracy

In [ ]:
# Create figure with 37 subplots
fig, axes = plt.subplots(nrows=10, ncols=4, figsize=(24, 50))  # 10x4 grid for 37 plots (with 3 empty)
axes = axes.ravel()  # Flatten for easy iteration

# Loop through each DataFrame
for i, data in enumerate(datasets):
    df = create_df(scores_dict, data, models)
    
    ax = axes[i]
    
    # Extract data
    cls = df.index
    means = df.iloc[:, 12]
    stds = df.iloc[:, 13]

    upper = df.iloc[0, 12] + df.iloc[0, 13]
    lower = df.iloc[0, 12] - df.iloc[0, 13]

    # Determine significant comparisons (if any) up front, so we can
    # reserve extra headroom above the data for the significance bars
    best_model = None
    sig_models = []
    if data in nemenyi_dict_ba:
        best_model = means.idxmax()
        posthoc = nemenyi_dict_ba[data]
        sig_models = [
            m for m in models
            if m != best_model and posthoc.loc[best_model, m] < 0.05
        ]

    # Create errorbar plot
    ax.errorbar(x=cls, y=means, yerr=stds,
                fmt='o', capsize=5, markersize=8)

    # Customize subplot
    ax.set_title(data, fontsize=20)
    ylims = compute_ylim(means, stds, upper, lower)
    if sig_models:
        # reserve headroom above the data for the significance bars
        y0, y1 = ylims
        ylims = (y0, y1 + (y1 - y0) * 0.4)
    ax.set_ylim(ylims)
    ax.tick_params(axis='x', rotation=90, labelsize=14)
    ax.grid(True, alpha=0.3)

    # Add horizontal lines at RF error bars
    ax.axhline(y=upper, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=lower, color='r', linestyle='--', alpha=0.5)

    # Color area between RF error bars
    ax.axhspan(ymin=lower, ymax=upper, facecolor='pink', alpha=0.3)

    # If the Friedman test was significant, draw Nemenyi significance bars
    # from the best performing model to each model significantly worse than it
    if sig_models:
        data_max = (means + stds).max()
        add_significance_bars(ax, models, data_max, best_model, sig_models)

# Hide unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.suptitle(f'Model Performance Across {len(datasets)} Datasets (Balanced accuracy)', fontsize=20, y=1.02)
plt.savefig('../figures/special-ensembles_balanced_acc.png', dpi=300, bbox_inches='tight')
plt.show()

## Brier Score Friedman test

In [ ]:
friedman_rows = []
nemenyi_dict_brier = {}

for data in datasets:
    df = create_fold_df(folds_dict, data, "brier")
    statistic, pvalue = apply_friedman_test(df)
    friedman_rows.append({"dataset": data, "statistic": statistic, "pvalue": pvalue})

    # follow up with Nemenyi post-hoc test only when Friedman is significant
    if pvalue < 0.05:
        nemenyi_dict_brier[data] = apply_nemenyi_test(df)

friedman_df_brier = pd.DataFrame(friedman_rows).set_index("dataset")
friedman_df_brier

## Brier score

In [ ]:
# Create figure with 37 subplots
fig, axes = plt.subplots(nrows=10, ncols=4, figsize=(24, 50))  # 10x4 grid for 37 plots (with 3 empty)
axes = axes.ravel()  # Flatten for easy iteration

# Loop through each DataFrame
for i, data in enumerate(datasets):
    df = create_df(scores_dict, data, models)
    
    ax = axes[i]
    
    # Extract data
    cls = df.index
    means = df.iloc[:, 14]
    stds = df.iloc[:, 15]

    upper = df.iloc[0, 14] + df.iloc[0, 15]
    lower = df.iloc[0, 14] - df.iloc[0, 15]

    # Determine significant comparisons (if any) up front, so we can
    # reserve extra headroom above the data for the significance bars
    best_model = None
    sig_models = []
    if data in nemenyi_dict_brier:
        best_model = means.idxmin()
        posthoc = nemenyi_dict_brier[data]
        sig_models = [
            m for m in models
            if m != best_model and posthoc.loc[best_model, m] < 0.05
        ]

    # Create errorbar plot
    ax.errorbar(x=cls, y=means, yerr=stds,
                fmt='o', capsize=5, markersize=8)

    # Customize subplot
    ax.set_title(data, fontsize=20)
    ylims = compute_ylim(means, stds, upper, lower)
    if sig_models:
        # reserve headroom above the data for the significance bars
        y0, y1 = ylims
        ylims = (y0, y1 + (y1 - y0) * 0.4)
    ax.set_ylim(ylims)
    ax.tick_params(axis='x', rotation=90, labelsize=14)
    ax.grid(True, alpha=0.3)

    # Add horizontal lines at RF error bars
    ax.axhline(y=upper, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=lower, color='r', linestyle='--', alpha=0.5)

    # Color area between RF error bars
    ax.axhspan(ymin=lower, ymax=upper, facecolor='pink', alpha=0.3)

    # If the Friedman test was significant, draw Nemenyi significance bars
    # from the best performing model to each model significantly worse than it
    if sig_models:
        data_max = (means + stds).max()
        add_significance_bars(ax, models, data_max, best_model, sig_models)

# Hide unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.suptitle(f'Model Performance Across {len(datasets)} Datasets (Brier Score)', fontsize=20, y=1.02)
plt.savefig('../figures/special-ensembles_brier.png', dpi=300, bbox_inches='tight')
plt.show()

## Join datasets and create table

In [ ]:
dfs = [create_df(scores_dict, data, models) for data in datasets]
names = datasets
result = pd.concat(dfs, keys=names, names=['dataset', 'model'])
result

In [ ]:
result['roc_±_std'] = result['roc'].round(3).astype(str) + '_±_' + result['roc_std'].round(3).astype(str)
result['ap_±_td'] = result['ap'].round(3).astype(str) + '_±_' + result['ap_std'].round(3).astype(str)
result['f1_±_std'] = result['f1_score'].round(3).astype(str) + '_±_' + result['f1_std'].round(3).astype(str)
result['mcc_±_td'] = result['mcc'].round(3).astype(str) + '_±_' + result['mcc_std'].round(3).astype(str)
result['ba_±_std'] = result['ba'].round(3).astype(str) + '_±_' + result['ba_std'].round(3).astype(str)
result['brier_±_td'] = result['brier'].round(3).astype(str) + '_±_' + result['brier_std'].round(3).astype(str)
result['gmean_±_std'] = result['gmean'].round(3).astype(str) + '_±_' + result['gmean_std'].round(3).astype(str)
result['thresh_±_std'] = result['thresh'].round(3).astype(str) + '_±_' + result['tresh_std'].round(3).astype(str)

result.to_csv('../results/special-ensembles_models_performance.csv', index=True)

result[['roc_±_std', 'ap_±_td', 'f1_±_std', 'mcc_±_td']]